# Chunking Strategy Benchmark — PDF Document (Capstone Proposal)

This notebook benchmarks chunking strategies for a **long, structured PDF document**
(a 32-page capstone proposal), using it as a stand-in for any narrative/report-style PDF
you'd ingest into a RAG pipeline.

A PDF is a different shape of data from the earlier decision-record experiment: it's one
long continuous document with a real physical structure (pages -> paragraphs -> sentences)
rather than short, already-atomic database rows. That's why **Parent-Child chunking**
(Page = parent, Paragraph = child) is the centerpiece strategy here — it's the standard
answer to PDFs' core tension: paragraphs give you precise retrieval, but a page (or
section) gives you the surrounding context needed to actually answer the question.

Strategies benchmarked:
1. **Whole-Page** — one chunk per PDF page (coarse baseline)
2. **Fixed-Size (RecursiveCharacterTextSplitter)** — naive character-budget splitting, applied per page
3. **Paragraph (flat)** — one chunk per reconstructed paragraph, no parent link — isolates whether "smaller chunks" alone helps
4. **Parent-Child (Page=Parent, Paragraph=Child)** — embed paragraphs for precise matching, but expand to the full page at answer time
5. **Semantic Chunking** — embedding-similarity based splits within each page

Embedding models compared: `all-MiniLM-L6-v2`, `BAAI/bge-small-en-v1.5`, `nomic-ai/nomic-embed-text-v1.5`.


In [1]:
# !pip install langchain-chroma


In [2]:
# !pip install langchain-experimental


In [3]:
# !pip install langchain-huggingface langchain-text-splitters sentence-transformers


In [4]:
# !pip install pdfplumber


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
import shutil
import os
import re
import json
import pandas as pd
import pdfplumber


## Extract the PDF: pages, then paragraphs within each page

`pdfplumber` gives clean per-page text. PDFs don't encode paragraphs directly — text
extraction returns one line per *visual* line, so a wrapped paragraph shows up as several
short lines that need to be re-joined. The heuristic below merges wrapped lines back into
paragraphs using two signals: a blank line always starts a new paragraph, and a
noticeably short line (well under the page's typical line width) that ends in sentence
punctuation is treated as the end of a paragraph even without a following blank line.

This is a heuristic, not a layout parser — production pipelines dealing with messier PDFs
(multi-column layouts, scanned pages, embedded tables) typically reach for a layout-aware
tool (PyMuPDF blocks, Unstructured.io, Docling) instead. It's good enough here because the
core thing being tested is the **Parent-Child indexing pattern**, not the extraction
quality — swap in a better paragraph extractor later without changing anything downstream.


In [6]:
PDF_PATH = "42_Capstone Proposal.pdf"


def clean_line(line):
    return re.sub(r"\s+", " ", line).strip()


def extract_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            pages.append({"page": i, "text": text})
    return pages


def text_to_paragraphs(text):
    raw_lines = text.split("\n")
    lines = [clean_line(l) for l in raw_lines]

    widths = [len(l) for l in lines if l]
    avg_width = sum(widths) / len(widths) if widths else 0
    short_line_threshold = avg_width * 0.65

    paragraphs = []
    current = []

    def flush():
        if current:
            paragraphs.append(" ".join(current).strip())
            current.clear()

    for line in lines:
        if not line:
            flush()
            continue

        current.append(line)

        looks_like_paragraph_end = (
            len(line) < short_line_threshold
            and line[-1] in ".!?:\u201d\")"
        )

        if looks_like_paragraph_end:
            flush()

    flush()

    # drop tiny fragments (running headers/footers/stray page numbers)
    paragraphs = [p for p in paragraphs if len(p) > 25]

    return paragraphs


pages = extract_pages(PDF_PATH)

for p in pages:
    p["paragraphs"] = text_to_paragraphs(p["text"])

# a page-number -> full page text lookup, used later for parent expansion
page_lookup = {p["page"]: p["text"].strip() for p in pages}

print(f"{len(pages)} pages extracted")
print(f"{sum(len(p['paragraphs']) for p in pages)} paragraphs reconstructed")
print()
print(pages[10]["paragraphs"][:2])  # sample from page 11


32 pages extracted
102 paragraphs reconstructed

['Colleges and other educational institutions often experience challenges in the reuse and preservation of strategic and operational decision-making knowledge. Information about past decisions made by the academic and administrative staff is mostly fragmented across different sources, including reports, policies, emails, and the knowledge held by the staff. This has made it difficult for these institutions to retain knowledge about past decisions, which has resulted in inconsistencies in the decision-making process. This has often come at a price, especially when the key staff members depart the institution. This is often referred to as corporate amnesia. (Wikipedia, 2024)', 'Evidence 1: 42% of Institutional Knowledge Is Unique to Each Individual Fact: When an employee quits their peers cannot perform 42 percent of their job due to the fact that such a person has a unique knowledge (Panopto, 2018) that he or she does not share with anybo

## Ground truth Q&A set

Hand-written questions against known facts in the document, each tagged with the page
(or pages, where the same fact is stated more than once) it should be retrievable from.
This lets every chunking strategy — page, paragraph, or sentence-level — be scored on a
common denominator: **did retrieval land on the right page(s)?**


In [7]:
ground_truth = [
{
    "question": "Who is the supervisor of this capstone project?",
    "ground_truth_answer": "Subit Timalsina",
    "expected_pages": [4],
    "answer_type": "Role"
},
{
    "question": "What group number is assigned to this project?",
    "ground_truth_answer": "Group 42",
    "expected_pages": [4],
    "answer_type": "Fact"
},
{
    "question": "What term describes the loss of institutional knowledge when key staff leave an organization?",
    "ground_truth_answer": "Corporate amnesia",
    "expected_pages": [11],
    "answer_type": "Concept"
},
{
    "question": "What percentage of institutional knowledge is unique to each individual, per the cited Panopto survey?",
    "ground_truth_answer": "42 percent",
    "expected_pages": [11],
    "answer_type": "Statistic"
},
{
    "question": "How many empirical studies were reviewed in the systematic review on knowledge loss from staff turnover?",
    "ground_truth_answer": "91 empirical studies",
    "expected_pages": [11],
    "answer_type": "Statistic"
},
{
    "question": "What frontend technologies make up the project's technology stack?",
    "ground_truth_answer": "React, TypeScript, and Vite",
    "expected_pages": [14],
    "answer_type": "Technical"
},
{
    "question": "Which three databases form the polyglot persistence layer?",
    "ground_truth_answer": "PostgreSQL, Neo4j, and pgvector",
    "expected_pages": [15],
    "answer_type": "Technical"
},
{
    "question": "What metrics will be used to evaluate the RAG system's performance?",
    "ground_truth_answer": "Precision and Recall at K",
    "expected_pages": [16],
    "answer_type": "Technical"
},
{
    "question": "What formula is used to calculate a risk score in the risk assessment methodology?",
    "ground_truth_answer": "Risk Score = Likelihood x Impact (R = L * I)",
    "expected_pages": [25],
    "answer_type": "Methodology"
},
{
    "question": "What risk score range is classified as High risk?",
    "ground_truth_answer": "10-16",
    "expected_pages": [25],
    "answer_type": "Methodology"
},
{
    "question": "What accuracy did the IKEDS framework achieve in decision-making tasks?",
    "ground_truth_answer": "85.7%",
    "expected_pages": [8, 10],
    "answer_type": "Statistic"
},
{
    "question": "What F1 score improvement did KG2RAG achieve over standard semantic retrieval?",
    "ground_truth_answer": "From 0.357 to 0.436",
    "expected_pages": [8, 10],
    "answer_type": "Statistic"
},
{
    "question": "What tools does the project team use for daily and weekly communication?",
    "ground_truth_answer": "GitHub, Google Meet, and WhatsApp",
    "expected_pages": [22],
    "answer_type": "Fact"
},
{
    "question": "Who holds the role of Project Lead, AI Architect and Backend Engineer?",
    "ground_truth_answer": "Samrat Jwala Uprety",
    "expected_pages": [22],
    "answer_type": "Role"
},
{
    "question": "Who is the Data Integration, Knowledge Graph Engineering and System Validation Lead?",
    "ground_truth_answer": "Sanjina Shrestha",
    "expected_pages": [23],
    "answer_type": "Role"
},
{
    "question": "Where will the frontend and backend of the system be deployed?",
    "ground_truth_answer": "Frontend on Vercel, backend on Render",
    "expected_pages": [15],
    "answer_type": "Technical"
},
]

print(len(ground_truth))


16


In [8]:
embedding_models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE": "BAAI/bge-small-en-v1.5",
    "Nomic": "nomic-ai/nomic-embed-text-v1.5"
}


In [9]:
def benchmark_embeddings(documents,
                          strategy_name,
                          persist_directory="vectorstores"):
    """
    documents:
        Output of your chunking function — list of dicts with
        "id", "text", "metadata". metadata must include "page".

    strategy_name:
        Example:
            WholePage
            FixedChunk
            ParagraphChunk
            ParentChild
            SemanticChunk
    """

    results = {}

    docs = [
        Document(
            page_content=d["text"],
            metadata=d["metadata"]
        )
        for d in documents
    ]

    for model_name, model in embedding_models.items():

        print("=" * 60)
        print(f"Chunking : {strategy_name}")
        print(f"Embedding: {model_name}")
        print("=" * 60)

        embedding = HuggingFaceEmbeddings(
            model_name=model,
            model_kwargs={
                "trust_remote_code": True
            }
        )

        db_path = os.path.join(
            persist_directory,
            f"{strategy_name}_{model_name}"
        )

        if os.path.exists(db_path):
            shutil.rmtree(db_path)

        vectordb = Chroma.from_documents(
            docs,
            embedding,
            persist_directory=db_path
        )

        retriever = vectordb.as_retriever(
            search_kwargs={
                "k": 3
            }
        )

        results[model_name] = retriever

        print("Done.\n")

    return results


In [10]:
def evaluate_retrievers(retrievers, strategy_name):
    """
    Runs the ground_truth set against every model's retriever for a given
    chunking strategy and returns a results DataFrame with Hit@1 and Hit@3,
    scored on page-number overlap (a chunk is "correct" if its page is one
    of the question's expected_pages).
    """

    rows = []

    for model_name, retriever in retrievers.items():

        for sample in ground_truth:

            question = sample["question"]
            expected_pages = set(sample["expected_pages"])

            docs = retriever.invoke(question)
            retrieved_pages = [d.metadata.get("page") for d in docs]

            hit_at_1 = bool(retrieved_pages) and retrieved_pages[0] in expected_pages
            hit_at_3 = any(p in expected_pages for p in retrieved_pages)

            rows.append({
                "Strategy": strategy_name,
                "Embedding": model_name,
                "Question": question,
                "Expected_Pages": sorted(expected_pages),
                "Top1_Page": retrieved_pages[0] if retrieved_pages else None,
                "Hit@1": hit_at_1,
                "Hit@3": hit_at_3
            })

    return pd.DataFrame(rows)


def summarize(results_df):
    summary = (
        results_df
        .groupby(["Strategy", "Embedding"])[["Hit@1", "Hit@3"]]
        .mean()
        .round(3) * 100
    )
    summary.columns = ["Hit@1 (%)", "Hit@3 (%)"]
    return summary


all_results = []  # collects one DataFrame per strategy for the final comparison


## Strategy 1 — Whole-Page Chunking

One chunk per PDF page.

**Why include it:** guarantees full local context every time — nothing on the page is
ever split away from its neighbors, and it's the cheapest strategy to index.

**Trade-off to watch:** pages here run 500-2800 characters and mix several topics (e.g.
page 22 covers both the communication plan *and* the roles table), so a narrow factual
question has to compete against everything else on the page inside one embedding. Expect
this to be the weakest strategy on `Hit@1` for the Statistic/Technical questions, even
though it should look "fine" on `Hit@3` since there's nowhere else for the right page to
hide.


In [11]:
def whole_page_chunking(pages):

    documents = []

    for p in pages:
        text = p["text"].strip()
        if not text:
            continue

        documents.append({
            "id": f'page_{p["page"]}',
            "text": text,
            "metadata": {
                "page": p["page"]
            }
        })

    return documents


documents = whole_page_chunking(pages)

print(len(documents))
print(documents[0])


32
{'id': 'page_1', 'text': 'CAPSTONE PROJECT\nPROPOSAL\nTitle: Development of a Web-Based Decision Support System for a Simulated\nOrganization Using Knowledge Graphs and Retrieval-Augmented Generation\nDate:\nFebruary - 2026', 'metadata': {'page': 1}}


In [12]:
whole_page_retrievers = benchmark_embeddings(
    documents,
    "WholePage"
)


Chunking : WholePage
Embedding: MiniLM
Done.

Chunking : WholePage
Embedding: BGE
Done.

Chunking : WholePage
Embedding: Nomic


<All keys matched successfully>


Done.



In [13]:
whole_page_results = evaluate_retrievers(whole_page_retrievers, "WholePage")
all_results.append(whole_page_results)

summarize(whole_page_results)


Hit@1 (%)  Hit@3 (%)
Strategy  Embedding                      
WholePage BGE             75.0       81.2
          MiniLM          75.0       87.5
          Nomic           75.0       81.2

## Strategy 2 — Fixed-Size Chunking (RecursiveCharacterTextSplitter)

Splits each page's text on a fixed character budget, applied **per page** so `page`
metadata stays accurate (splitting the whole 32-page document as one string would lose
page attribution).

**Why include it:** the default most people reach for first — worth confirming whether
it earns that default status on this document rather than assuming it.

**Trade-off to watch:** with no awareness of paragraph or sentence boundaries, a chunk
boundary can land mid-sentence or mid-fact (e.g. splitting "85.7 percent" from the
framework name it describes), which can hurt precision in ways that are easy to miss
until you look at Hit@1 broken down by question.


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def fixed_chunking(pages, chunk_size=350, chunk_overlap=50):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    documents = []

    for p in pages:
        text = p["text"].strip()
        if not text:
            continue

        chunks = splitter.split_text(text)

        for i, chunk in enumerate(chunks):

            documents.append({
                "id": f'page_{p["page"]}_chunk_{i}',
                "text": chunk,
                "metadata": {
                    "page": p["page"]
                }
            })

    return documents


documents = fixed_chunking(pages)

print(len(documents))
print(documents[:2])


205
[{'id': 'page_1_chunk_0', 'text': 'CAPSTONE PROJECT\nPROPOSAL\nTitle: Development of a Web-Based Decision Support System for a Simulated\nOrganization Using Knowledge Graphs and Retrieval-Augmented Generation\nDate:\nFebruary - 2026', 'metadata': {'page': 1}}, {'id': 'page_2_chunk_0', 'text': 'Contents\nA. 4\nB. 5\n1. Review of Literature ............................................................................................................... 5\n1.1 The Challenge of Organizational Knowledge Loss .......................................................... 5', 'metadata': {'page': 2}}]


In [15]:
fixed_retrievers = benchmark_embeddings(
    documents,
    "FixedChunk"
)


Chunking : FixedChunk
Embedding: MiniLM
Done.

Chunking : FixedChunk
Embedding: BGE
Done.

Chunking : FixedChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [16]:
fixed_results = evaluate_retrievers(fixed_retrievers, "FixedChunk")
all_results.append(fixed_results)

summarize(fixed_results)


Hit@1 (%)  Hit@3 (%)
Strategy   Embedding                      
FixedChunk BGE             75.0       87.5
           MiniLM          81.2       87.5
           Nomic           87.5       87.5

## Strategy 3 — Paragraph Chunking (flat)

One chunk per reconstructed paragraph, with **no parent link** — each paragraph stands
alone with just its page number as metadata.

**Why include it:** isolates one variable at a time. Strategy 4 (Parent-Child) also
indexes at paragraph granularity, so comparing 3 against 4 tells you specifically what
the *parent expansion* is buying you, separate from what smaller-chunk-size alone buys
you.

**Trade-off to watch:** should win (or tie) on `Hit@1` versus Whole-Page and Fixed-Size,
since paragraphs are usually single-topic. But a retrieved paragraph in isolation can
lack context a reader (or an LLM generating an answer) would want — e.g. a paragraph
that says "This DSS was evaluated by comparing..." without repeating what "This DSS"
refers to.


In [17]:
def paragraph_chunking(pages):

    documents = []

    for p in pages:
        for i, para in enumerate(p["paragraphs"]):

            documents.append({
                "id": f'page_{p["page"]}_para_{i}',
                "text": para,
                "metadata": {
                    "page": p["page"]
                }
            })

    return documents


documents = paragraph_chunking(pages)

print(len(documents))
print(documents[:3])


102
[{'id': 'page_1_para_0', 'text': 'CAPSTONE PROJECT PROPOSAL Title: Development of a Web-Based Decision Support System for a Simulated Organization Using Knowledge Graphs and Retrieval-Augmented Generation Date:', 'metadata': {'page': 1}}, {'id': 'page_2_para_0', 'text': 'Contents A. 4 B. 5 1. Review of Literature ............................................................................................................... 5 1.1 The Challenge of Organizational Knowledge Loss .......................................................... 5 1.2 Decision Support Systems in Organizations ..................................................................... 6 1.3 Limitations of Traditional Decision Support Systems ...................................................... 7 1.4 Knowledge Graphs as Structured Knowledge Representation ......................................... 7 1.5 Retrieval-Augmented Generation for Grounded Responses ............................................. 7 1.6 Combining 

In [18]:
paragraph_retrievers = benchmark_embeddings(
    documents,
    "ParagraphChunk"
)


Chunking : ParagraphChunk
Embedding: MiniLM
Done.

Chunking : ParagraphChunk
Embedding: BGE
Done.

Chunking : ParagraphChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [19]:
paragraph_results = evaluate_retrievers(paragraph_retrievers, "ParagraphChunk")
all_results.append(paragraph_results)

summarize(paragraph_results)


Hit@1 (%)  Hit@3 (%)
Strategy       Embedding                      
ParagraphChunk BGE             50.0       87.5
               MiniLM          50.0       75.0
               Nomic           68.8       93.8

## Strategy 4 — Parent-Child Chunking (Page = Parent, Paragraph = Child)

This is the strategy you asked to test. The **child** (paragraph) is what gets embedded
and matched against the query — same granularity as Strategy 3. What's different is what
happens *after* a match: instead of handing the isolated paragraph to the LLM, you look
up its parent page and hand over the **full page** as context.

Two ways to implement this, and the trade-off between them is worth knowing before you
pick one for production:

- **Metadata-embedded parent** (what Strategy 3 vs. Whole-Page essentially compares):
  store the full parent text inside every child's vector-store metadata. Simple, but
  duplicates the parent text once per child paragraph on that page — wasteful at scale.
- **Docstore-backed parent** (what's implemented below, and what LangChain's
  `ParentDocumentRetriever` does): only the paragraph is embedded; the parent page text
  lives once in a separate lookup keyed by `page` number, fetched *after* retrieval.
  This is the version you want in production — it keeps the vector store lean and lets
  you swap in a smarter parent unit (e.g. a whole *section* instead of a page) without
  re-embedding anything.

Because retrieval itself is scored on the same paragraph-level match as Strategy 3, this
strategy's `Hit@1`/`Hit@3` numbers will be **identical** to Strategy 3 — that's expected
and correct. The benefit of Parent-Child isn't a different retrieval score, it's a
**better-grounded generation step** afterward. The comparison below demonstrates the
expansion explicitly so you can see what the LLM would actually receive.


In [20]:
def parent_child_chunking(pages):
    """
    Indexes at paragraph (child) granularity, identical to paragraph_chunking.
    The parent page text is *not* duplicated into metadata — it is looked up
    separately via `page_lookup` at answer time (see expand_to_parent below).
    """

    documents = []

    for p in pages:
        for i, para in enumerate(p["paragraphs"]):

            documents.append({
                "id": f'page_{p["page"]}_para_{i}',
                "text": para,
                "metadata": {
                    "page": p["page"],
                    "chunk_type": "child_paragraph"
                }
            })

    return documents


def expand_to_parent(retrieved_docs, page_lookup):
    """
    Given child (paragraph) chunks returned by the retriever, returns the
    full parent page text for each one -- this is what would actually get
    passed to the LLM as context, instead of the bare paragraph.
    """

    expanded = []

    for doc in retrieved_docs:
        page_num = doc.metadata.get("page")
        expanded.append({
            "matched_paragraph": doc.page_content,
            "page": page_num,
            "parent_context": page_lookup.get(page_num, "")
        })

    return expanded


documents = parent_child_chunking(pages)

print(len(documents))
print(documents[:2])


102
[{'id': 'page_1_para_0', 'text': 'CAPSTONE PROJECT PROPOSAL Title: Development of a Web-Based Decision Support System for a Simulated Organization Using Knowledge Graphs and Retrieval-Augmented Generation Date:', 'metadata': {'page': 1, 'chunk_type': 'child_paragraph'}}, {'id': 'page_2_para_0', 'text': 'Contents A. 4 B. 5 1. Review of Literature ............................................................................................................... 5 1.1 The Challenge of Organizational Knowledge Loss .......................................................... 5 1.2 Decision Support Systems in Organizations ..................................................................... 6 1.3 Limitations of Traditional Decision Support Systems ...................................................... 7 1.4 Knowledge Graphs as Structured Knowledge Representation ......................................... 7 1.5 Retrieval-Augmented Generation for Grounded Responses .............................

In [21]:
parent_child_retrievers = benchmark_embeddings(
    documents,
    "ParentChild"
)


Chunking : ParentChild
Embedding: MiniLM
Done.

Chunking : ParentChild
Embedding: BGE
Done.

Chunking : ParentChild
Embedding: Nomic


<All keys matched successfully>


Done.



In [22]:
parent_child_results = evaluate_retrievers(parent_child_retrievers, "ParentChild")
all_results.append(parent_child_results)

summarize(parent_child_results)


Hit@1 (%)  Hit@3 (%)
Strategy    Embedding                      
ParentChild BGE             50.0       87.5
            MiniLM          50.0       75.0
            Nomic           68.8       93.8

In [25]:
# Demonstration: what the LLM actually receives with parent expansion,
# versus the bare paragraph a flat paragraph-chunk strategy would hand over.

demo_question = "What accuracy did the IKEDS framework achieve in decision-making tasks?"
demo_retriever = parent_child_retrievers["BGE"]

matched_children = demo_retriever.invoke(demo_question)
expanded = expand_to_parent(matched_children, page_lookup)

for item in expanded[:1]:
    print("Matched paragraph (child):\n", item["matched_paragraph"])
    print("\n" + "-" * 60 + "\n")
    print(f"Expanded parent context (page {item['page']}):\n", item["parent_context"][:600])


Matched paragraph (child):
 Wang et al. (2025) – Introduces a smart DSS This framework is extended into our Construction of architecture that combines AI project where modern LLM-based Intelligent Decision techniques with knowledge RAG architecture and Knowledge Support Systems integration and proves to be Graphs are incorporated to increase (IKEDS) very accurate across a variety the explainability, scalability, and of fields. domain adaptability in a simulated organizational environment.

------------------------------------------------------------

Expanded parent context (page 29):
 Generation for LLMs: A single document fragments relationship awareness, and multi-
Survey without recording the semantic hop reasoning.
connections between them.
Wang et al. (2025) – Introduces a smart DSS This framework is extended into our
Construction of architecture that combines AI project where modern LLM-based
Intelligent Decision techniques with knowledge RAG architecture and Knowledge
Support S

## Strategy 5 — Semantic Chunking

Uses `SemanticChunker` (embedding-similarity breakpoints), applied **per page** so a
detected "topic shift" can't accidentally straddle a page boundary and lose the `page`
metadata's meaning.

**Why include it:** semantic chunking is often pitched as strictly better than
fixed-size splitting because chunk boundaries land on actual topic shifts rather than an
arbitrary character count. Worth testing directly against Paragraph/Parent-Child chunking
rather than assuming it wins by default.

**Trade-off to watch:** on a document that already has real paragraph structure (like
this proposal), semantic breakpoints will often converge on roughly the same boundaries a
human paragraph break already gives you for free — at the added cost of an embedding
call per page during chunking itself. Where it can outperform paragraph chunking is on
pages containing multiple short, choppy paragraphs on the same topic that a paragraph
splitter would keep separate but that arguably *should* be merged.


In [26]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

semantic_splitter = SemanticChunker(semantic_embedding)


def semantic_chunking(pages):

    documents = []

    for p in pages:
        text = p["text"].strip()
        if not text:
            continue

        chunks = semantic_splitter.split_text(text)

        for i, chunk in enumerate(chunks):

            documents.append({
                "id": f'page_{p["page"]}_sem_{i}',
                "text": chunk,
                "metadata": {
                    "page": p["page"]
                }
            })

    return documents


documents = semantic_chunking(pages)

print(len(documents))
print(documents[:2])


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27052\2704677394.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


68
[{'id': 'page_1_sem_0', 'text': 'CAPSTONE PROJECT\nPROPOSAL\nTitle: Development of a Web-Based Decision Support System for a Simulated\nOrganization Using Knowledge Graphs and Retrieval-Augmented Generation\nDate:\nFebruary - 2026', 'metadata': {'page': 1}}, {'id': 'page_2_sem_0', 'text': 'Contents\nA. 4\nB.', 'metadata': {'page': 2}}]


In [27]:
semantic_retrievers = benchmark_embeddings(
    documents,
    "SemanticChunk"
)


Chunking : SemanticChunk
Embedding: MiniLM
Done.

Chunking : SemanticChunk
Embedding: BGE
Done.

Chunking : SemanticChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [28]:
semantic_results = evaluate_retrievers(semantic_retrievers, "SemanticChunk")
all_results.append(semantic_results)

summarize(semantic_results)


Hit@1 (%)  Hit@3 (%)
Strategy      Embedding                      
SemanticChunk BGE             62.5       81.2
              MiniLM          50.0       81.2
              Nomic           56.2       81.2

## Final comparison — all strategies x all models

`Hit@1` = the top retrieved chunk's page is one of the question's expected pages.
`Hit@3` = a correct page appears anywhere in the top 3 retrieved chunks.


In [29]:
master_results = pd.concat(all_results, ignore_index=True)

final_summary = summarize(master_results)
final_summary


Hit@1 (%)  Hit@3 (%)
Strategy       Embedding                      
FixedChunk     BGE             75.0       87.5
               MiniLM          81.2       87.5
               Nomic           87.5       87.5
ParagraphChunk BGE             50.0       87.5
               MiniLM          50.0       75.0
               Nomic           68.8       93.8
ParentChild    BGE             50.0       87.5
               MiniLM          50.0       75.0
               Nomic           68.8       93.8
SemanticChunk  BGE             62.5       81.2
               MiniLM          50.0       81.2
               Nomic           56.2       81.2
WholePage      BGE             75.0       81.2
               MiniLM          75.0       87.5
               Nomic           75.0       81.2

In [30]:
pivot_hit1 = final_summary["Hit@1 (%)"].unstack("Embedding")
pivot_hit3 = final_summary["Hit@3 (%)"].unstack("Embedding")

print("Hit@1 (%) by strategy x embedding model")
display(pivot_hit1)

print("\nHit@3 (%) by strategy x embedding model")
display(pivot_hit3)


Hit@1 (%) by strategy x embedding model


Embedding,BGE,MiniLM,Nomic
Strategy,,,
FixedChunk,75.0,81.2,87.5
ParagraphChunk,50.0,50.0,68.8
ParentChild,50.0,50.0,68.8
SemanticChunk,62.5,50.0,56.2
WholePage,75.0,75.0,75.0



Hit@3 (%) by strategy x embedding model


Embedding,BGE,MiniLM,Nomic
Strategy,,,
FixedChunk,87.5,87.5,87.5
ParagraphChunk,87.5,75.0,93.8
ParentChild,87.5,75.0,93.8
SemanticChunk,81.2,81.2,81.2
WholePage,81.2,87.5,81.2


## Reading the results / recommendation

Fill in the specific numbers after running, but expect (and verify):

- **Paragraph and Parent-Child should tie each other** on `Hit@1`/`Hit@3` — by design,
  since Parent-Child retrieves at the same paragraph granularity. If they *don't* match
  in your run, that's a bug in the chunking/metadata, not a real strategy difference.
- **Paragraph/Parent-Child should beat Whole-Page** on `Hit@1` for the Statistic/Technical
  questions specifically (e.g. "85.7%", "10-16", "PostgreSQL, Neo4j, and pgvector") —
  those answers are one sentence buried inside a multi-topic page, which is exactly the
  dilution problem narrow chunking fixes.
- **Whole-Page should have the smallest Hit@1-to-Hit@3 gap** — there's only one chunk per
  page, so if the right page is "in the running" at all it's usually the top hit too.
- **Fixed-Size should sit between Whole-Page and Paragraph** — it gets some of the
  precision benefit of smaller chunks, undercut by boundaries that don't respect
  sentence/paragraph structure.
- **Semantic chunking's edge over Paragraph chunking, if any, should be small** on a
  document with clear paragraph structure like this one — its value proposition matters
  more on messier or more free-flowing source text.

**Practical recommendation for this kind of document:** index at **paragraph
granularity** for retrieval precision, but always carry `page` (or a finer `section` id,
if you extend the extractor to detect headings) in metadata, and implement the
**docstore-backed Parent-Child pattern** shown in Strategy 4 — keep a separate
`page_lookup` (or LangChain's `ParentDocumentRetriever` + `InMemoryStore`) rather than
duplicating parent text into every child's metadata. That gets you precise matching *and*
a well-grounded context window for generation, without bloating the vector store.

For the **embedding model**, look at which one is most consistently strong across
strategies (not just its best case) in the pivot tables above — the model you deploy
should hold up across whatever chunking strategy your pipeline ends up using, not just
the one it happened to win on here.
